In [ ]:
# Default parameters for manual testing (ADF will inject values here)
entity_name = "ALL"          # Options: "ALL", "suppliers", "purchase_orders", "deliveries", "inventory"
processing_date = "2026-07-22"

In [ ]:
# ------------------------------------------------------------------
# Synapse / Fabric Notebook: Transform_Bronze_To_Silver
# ------------------------------------------------------------------
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)
from pyspark.sql.functions import (
    col, trim, initcap, lower, upper, when, lit, to_date
)

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Storage configuration
storage_account = "stlakesupply"
container = "medallion"

# Extract date components from parameter injected above
year, month, day = processing_date.split("-")

bronze_base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/bronze"
silver_base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/silver"

# ------------------------------------------------------------------
# 1. Schema definitions for all entities
# ------------------------------------------------------------------
schema_map = {
    "suppliers": StructType([
        StructField("supplier_id", StringType(), True),
        StructField("supplier_name", StringType(), True),
        StructField("country", StringType(), True),
        StructField("region", StringType(), True),
        StructField("category", StringType(), True),
        StructField("lead_time_days", IntegerType(), True),
        StructField("quality_score", DoubleType(), True)
    ]),
    "purchase_orders": StructType([
        StructField("order_id", StringType(), True),
        StructField("supplier_id", StringType(), True),
        StructField("product_id", StringType(), True),
        StructField("warehouse_id", StringType(), True),
        StructField("order_date", DateType(), True),
        StructField("quantity_ordered", IntegerType(), True),
        StructField("unit_cost_eur", DoubleType(), True),
        StructField("total_value_eur", DoubleType(), True),
        StructField("status", StringType(), True)
    ]),
    "deliveries": StructType([
        StructField("delivery_id", StringType(), True),
        StructField("order_id", StringType(), True),
        StructField("actual_delivery_date", DateType(), True),
        StructField("quantity_delivered", IntegerType(), True),
        StructField("quantity_accepted", IntegerType(), True),
        StructField("quantity_rejected", IntegerType(), True),
        StructField("delivery_status", StringType(), True),
        StructField("carrier", StringType(), True)
    ]),
    "inventory": StructType([
        StructField("warehouse_id", StringType(), True),
        StructField("product_id", StringType(), True),
        StructField("quantity_on_hand", IntegerType(), True)
    ])
}

# Determine entities to process based on ADF parameter
if entity_name.upper() == "ALL":
    entities_to_process = list(schema_map.keys())
else:
    if entity_name not in schema_map:
        raise ValueError(f"Unknown entity '{entity_name}'. Must be one of {list(schema_map.keys())} or 'ALL'.")
    entities_to_process = [entity_name]

print(f"--- Starting Silver Ingestion for Date: {processing_date} ---")
print(f"Target entity parameter: {entity_name}")
print(f"Entities queued ({len(entities_to_process)}): {', '.join(entities_to_process)}\n")

# ------------------------------------------------------------------
# 2. Iterative Pipeline Execution
# ------------------------------------------------------------------
for current_entity in entities_to_process:
    print("=" * 60)
    print(f" Processing Entity: {current_entity}")
    print("=" * 60)
    
    bronze_path = f"{bronze_base_path}/{current_entity}.csv"
    silver_path = f"{silver_base_path}/{current_entity}"
    schema = schema_map[current_entity]

    try:
        # A. Read CSV from Bronze
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "false") \
            .schema(schema) \
            .load(bronze_path) 

        print(f"  [+] Loaded {df.count()} raw rows from Bronze")

        # B. Trim string columns
        for field in schema.fields:
            if isinstance(field.dataType, StringType):
                df = df.withColumn(field.name, trim(col(field.name)))

        # C. Entity-specific transformations
        if current_entity == "suppliers":
            df = df.withColumn("supplier_name", initcap(col("supplier_name"))) \
                   .withColumn("category", lower(col("category"))) \
                   .withColumn("country", upper(col("country"))) \
                   .withColumn("region", upper(col("region")))
            df = df.withColumn("quality_score", when(col("quality_score").isNull(), 0.0)
                                               .otherwise(col("quality_score")))
            natural_key = "supplier_id"

        elif current_entity == "purchase_orders":
            df = df.withColumn("order_id", upper(col("order_id"))) \
                   .withColumn("supplier_id", upper(col("supplier_id"))) \
                   .withColumn("product_id", upper(col("product_id"))) \
                   .withColumn("warehouse_id", upper(col("warehouse_id"))) \
                   .withColumn("status", initcap(col("status")))
            df = df.withColumn("quantity_ordered", when(col("quantity_ordered") < 0, 0)
                                               .otherwise(col("quantity_ordered")))
            df = df.withColumn("total_value_eur",
                               when(col("total_value_eur").isNull(),
                                    col("quantity_ordered") * col("unit_cost_eur"))
                               .otherwise(col("total_value_eur")))
            natural_key = "order_id"

        elif current_entity == "deliveries":
            df = df.withColumn("delivery_id", upper(col("delivery_id"))) \
                   .withColumn("order_id", upper(col("order_id"))) \
                   .withColumn("carrier", initcap(col("carrier"))) \
                   .withColumn("delivery_status", initcap(col("delivery_status")))
            df = df.withColumn("quantity_delivered",
                               when(col("quantity_delivered") < (col("quantity_accepted") + col("quantity_rejected")),
                                    col("quantity_accepted") + col("quantity_rejected"))
                               .otherwise(col("quantity_delivered")))
            natural_key = "delivery_id"

        elif current_entity == "inventory":
            df = df.withColumn("warehouse_id", upper(col("warehouse_id"))) \
                   .withColumn("product_id", upper(col("product_id")))
            df = df.withColumn("quantity_on_hand", when(col("quantity_on_hand") < 0, 0)
                                               .otherwise(col("quantity_on_hand")))
            natural_key = ["warehouse_id", "product_id"]

        # D. Deduplication
        dedup_keys = natural_key if isinstance(natural_key, list) else [natural_key]
        df_clean = df.dropDuplicates(dedup_keys)
        print(f"  [+] Deduplicated rows: {df_clean.count()}")

        # E. Add partition metadata
        df_final = df_clean.withColumn("processing_date", to_date(lit(processing_date))) \
                           .withColumn("year", lit(year)) \
                           .withColumn("month", lit(month)) \
                           .withColumn("day", lit(day))

        df_final = df_final.repartition("year", "month", "day")

        # F. Write to Silver (date-partitioned Parquet)
        if current_entity == "inventory":
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
            df_final.write.mode("overwrite") \
                .partitionBy("year", "month", "day") \
                .parquet(silver_path)
        else:
            df_final.write.mode("append") \
                .partitionBy("year", "month", "day") \
                .parquet(silver_path)

        print(f"  [✓] Silver write complete for {current_entity}.\n")

    except Exception as e:
        print(f"  [✗] Error processing {current_entity}: {str(e)}\n")

print("=" * 60)
print("Processing execution finished.")